In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

In [2]:
url = 'https://statusinvest.com.br/category/advancedsearchresultpaginated?search=%7B%22Sector%22%3A%22%22%2C%22SubSector%22%3A%22%22%2C%22Segment%22%3A%22%22%2C%22my_range%22%3A%22-20%3B100%22%2C%22forecast%22%3A%7B%22upsidedownside%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22estimatesnumber%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22revisedup%22%3Atrue%2C%22reviseddown%22%3Atrue%2C%22consensus%22%3A%5B%5D%7D%2C%22dy%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_l%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22peg_ratio%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_vp%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_ativo%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22margembruta%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22margemebit%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22margemliquida%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_ebit%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22ev_ebit%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22dividaliquidaebit%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22dividaliquidapatrimonioliquido%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_sr%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_capitalgiro%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_ativocirculante%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22roe%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22roic%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22roa%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22liquidezcorrente%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22pl_ativo%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22passivo_ativo%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22giroativos%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22receitas_cagr5%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22lucros_cagr5%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22liquidezmediadiaria%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22vpa%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22lpa%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22valormercado%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%7D&orderColumn=&isAsc=&page=0&take=617&CategoryType=1'
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
data = response.json()
indicadores = pd.DataFrame(data["list"])

In [3]:
screening = indicadores[['companyname', 'ticker', 'price','dy', 'p_l', 'margemliquida', 'dividaliquidaebit','roe', 'roic','liquidezmediadiaria']]

In [4]:
#Tirando as empresas com liquidez diária menor que 1.000.000
screening = screening[screening['liquidezmediadiaria'] >=1_000_000]

In [5]:
# Filtro de Joel Greenblatt pelo ROIC
screening = screening[screening['roic'] >=8 ]

In [6]:
# Filtro de Dividend Yield pelo Décio Bazin
screening = screening[screening['dy'] >=5]

In [7]:
# Filtro para não pagar caro na ação
screening = screening[screening['p_l'] <=15]

In [8]:
# usar bom senso qualitativo
screening = screening[screening['margemliquida'] >=5]

In [9]:
screening = screening[screening['dividaliquidaebit'] <=5]

In [10]:
screening = screening[screening['roe'] >=9.5]

In [11]:
# Ranquear Dividend Yield do maior para o menor
screening = screening.sort_values("dy", ascending=False)
screening["ranking_dy"] = range(1, len(screening) + 1)

In [12]:
# Ranquear P/L do menor para o maior
screening = screening.sort_values("p_l")
screening["ranking_p/l"] = range(1, len(screening) + 1)

In [13]:
# Ranquear Margem Líquida do maior para o menor
screening = screening.sort_values("margemliquida", ascending=False)
screening["ranking_marg_liq"] = range(1, len(screening) + 1)

In [14]:
# Ranquear Dívida liquida sobre EBIT do menor para o maior
screening = screening.sort_values("dividaliquidaebit")
screening["ranking_divida_liquida/ebit"] = range(1, len(screening) + 1)

In [15]:
# Ranquear ROE do maior para o menor
screening = screening.sort_values("roe", ascending=False)
screening["ranking_roe"] = range(1, len(screening) + 1)

In [16]:
screening["score_final"] = screening[["ranking_dy", "ranking_p/l", "ranking_marg_liq", "ranking_divida_liquida/ebit",'ranking_roe' ]].sum(axis=1)

In [17]:
screening = screening.sort_values("score_final")

Com essses filtros, é possível escolher quais ações GERAIS analisar os relatórios de forma performática!

Depois disso, usar o preço teto projetivo do Bazin nessas ações para saber o quão caro está

In [18]:
screening = screening[['companyname', 'ticker', 'price','dy', 'p_l', 'margemliquida', 'dividaliquidaebit','roe', 'roic','liquidezmediadiaria']]

In [87]:
tickers = screening["ticker"].tolist()
def get_setor(ticker):

    url = f"https://www.fundamentus.com.br/detalhes.php?papel={ticker}"

    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.text, "html.parser")
    links = soup.find_all("a", href=True)
    setor = None
    for link in links:
            href = link["href"]

            if "resultado.php?segmento=" in href:
                if not setor:
                    setor = link.text.strip()
                return {
        "ticker": ticker,
        "setor": setor,
    }

In [ ]:
resultados = []
for ticker in tickers:
    print(f"Buscando {ticker}...")
    resultados.append(get_setor(ticker))
    time.sleep(0.25)  # evita bloqueio

setores = pd.DataFrame(resultados)

In [ ]:
screening = screening.merge(setores, on="ticker", how="left")

In [96]:
coluna_setor = screening.pop("setor")
posicao = screening.columns.get_loc("companyname") + 1
screening.insert(posicao, "setor", coluna_setor)

In [97]:
screening

,companyname,setor,ticker,price,dy,p_l,margemliquida,dividaliquidaebit,roe,roic,liquidezmediadiaria
0,VULCABRAS/AZALEIA S.A.,Calçados,VULC3,15.80,31.36,4.31,32.73,1.03,48.01,12.88,1.186114e+07
1,GRENDENE S.A.,Calçados,GRND3,4.38,35.16,6.13,24.95,-3.04,20.44,8.04,1.426298e+07
2,LOJAS RIACHUELO S.A.,"Tecidos, Vestuário e Calçados",RIAA3,9.54,24.81,3.25,14.05,0.06,27.56,25.71,1.182586e+07
3,LAVVI EMPREENDIMENTOS IMOBILIÁRIOS S.A.,Incorporações,LAVV3,13.05,20.53,6.16,23.51,0.85,23.25,13.52,7.938584e+06
4,DIRECIONAL ENGENHARIA S.A.,Incorporações,DIRR3,12.86,17.24,8.48,18.18,0.66,34.02,17.50,7.363133e+07
5,MOURA DUBEUX ENGENHARIA S/A,Incorporações,MDNE3,30.19,17.73,7.48,17.84,0.73,27.72,16.12,3.206189e+07
6,CURY CONSTRUTORA E INCORPORADORA S.A.,Incorporações,CURY3,30.05,14.02,9.49,18.07,-0.25,58.69,35.99,1.009208e+08
7,SCHULZ S.A.,"Motores, Compressores e Outros",SHUL4,5.21,12.08,6.44,14.91,-0.56,19.27,9.24,1.945822e+06
8,ALLIED TECNOLOGIA S.A.,Eletrodomésticos,ALLD3,6.25,18.54,1.80,6.04,0.08,21.44,17.95,2.562289e+06
9,MARCOPOLO S.A.,Material Rodoviário,POMO3,6.44,14.67,6.58,13.51,1.09,31.40,12.17,5.023712e+06
